In [1]:
import re
import pandas as pd

log_pattern = re.compile(
    r'(\S+) (\S+) (\S+) \[(.*?)\] "(.*?)" (\d{3}) (\S+)'
)

rows = []

with open("data/raw/finalweblog.txt", "r") as f:
    for line in f:
        m = log_pattern.match(line)
        if m:
            rows.append(m.groups())

df = pd.DataFrame(rows, columns=[
    "ip","ident","authuser","timestamp",
    "request","status","bytes"
])

print("Rows parsed:", len(df))

df.to_csv("parsed_logs.csv", index=False)
print("Saved → data/processed/parsed_logs.csv")

Rows parsed: 100000
Saved → data/processed/parsed_logs.csv


In [2]:
# -------------------------
# file for split load and clean 
# -------------------------
import pandas as pd

# -------------------------
# Load parsed logs
# -------------------------
df = pd.read_csv("parsed_logs.csv")
print("Before split:", len(df))

# -------------------------
# SAFE REQUEST SPLIT
# -------------------------
req_parts = df['request'].str.split(' ', n=2, expand=True)

df['method'] = req_parts[0]
df['url'] = req_parts[1]
df['protocol'] = req_parts[2]

# remove broken request rows
df = df.dropna(subset=['method','url','protocol'])

# -------------------------
# FIX PROTOCOL VALUES
# keep only real HTTP/x.x
# -------------------------
df['protocol'] = df['protocol'].str.extract(
    r'(HTTP/\d\.\d)', expand=False
)

df = df.dropna(subset=['protocol'])

# -------------------------
# CLEAN TYPES
# -------------------------
df['bytes'] = df['bytes'].replace('-', 0)
df['bytes'] = df['bytes'].astype(int)

df['status'] = df['status'].astype(int)

# -------------------------
# TIMESTAMP FEATURES
# -------------------------
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    format="%d/%b/%Y:%H:%M:%S %z"
)

df['hour'] = df['timestamp'].dt.hour
df['day'] = df['timestamp'].dt.day

# -------------------------
# DROP UNUSED COLUMNS
# -------------------------
df = df.drop(columns=[
    'request',
    'ident',
    'authuser',
    'timestamp'
])

print("After clean:", len(df))
print("Protocol values:", df['protocol'].unique())

# -------------------------
# SAVE FINAL CLEAN FILE
# -------------------------
df.to_csv("clean_logs.csv", index=False)
print("Saved → clean_logs.csv")

Before split: 100000
After clean: 99768
Protocol values: ['HTTP/1.0']
Saved → clean_logs.csv


In [4]:
# -----------------------------
# ENCODING FILE 
# -----------------------------

import pandas as pd
import hashlib
from sklearn.preprocessing import LabelEncoder

# -----------------------------
# LOAD CLEAN DATA
# -----------------------------
df = pd.read_csv("clean_logs.csv")

print("Rows before encoding:", len(df))

# ensure numeric
df['status'] = df['status'].astype(int)

# -----------------------------
# URL FEATURE ENGINEERING
# -----------------------------
df['url_len'] = df['url'].str.len()
df['url_depth'] = df['url'].str.count('/')

df['is_image'] = df['url'].str.contains(
    r'\.gif|\.jpg|\.png', case=False, regex=True
).astype(int)

df['is_html'] = df['url'].str.contains(
    r'\.html', case=False, regex=True
).astype(int)

# -----------------------------
# METHOD & PROTOCOL ONE-HOT
# -----------------------------
df = pd.get_dummies(df, columns=['method', 'protocol'])

# -----------------------------
# STABLE IP HASH ENCODING
# -----------------------------
df['ip_enc'] = df['ip'].apply(
    lambda x: int(hashlib.md5(x.encode()).hexdigest(), 16) % 10000
)

df.drop(columns=['ip'], inplace=True)

# -----------------------------
# URL LABEL ENCODING
# -----------------------------
le = LabelEncoder()
df['url_enc'] = le.fit_transform(df['url'])

df.drop(columns=['url'], inplace=True)

# -----------------------------
# TARGET COLUMN
# -----------------------------
df['is_error'] = (df['status'] >= 400).astype(int)

# -----------------------------
# SAVE
# -----------------------------
print("Final columns:", df.columns.tolist())
print("\nSample rows:")
print(df.head())

df.to_csv("encoded_logs.csv", index=False)
print("\nSaved → encoded_logs.csv")

Rows before encoding: 99768
Final columns: ['status', 'bytes', 'hour', 'day', 'url_len', 'url_depth', 'is_image', 'is_html', 'method_GET', 'method_HEAD', 'method_POST', 'protocol_HTTP/1.0', 'ip_enc', 'url_enc', 'is_error']

Sample rows:
   status  bytes  hour  day  url_len  url_depth  is_image  is_html  \
0     200   6245     0    1       16          3         0        0   
1     200   3985     0    1       19          3         0        0   
2     200   4085     0    1       44          4         0        1   
3     304      0     0    1       31          3         0        1   
4     200   4179     0    1       47          4         1        0   

   method_GET  method_HEAD  method_POST  protocol_HTTP/1.0  ip_enc  url_enc  \
0        True        False        False               True    4029     2054   
1        True        False        False               True    8049     3059   
2        True        False        False               True    3990     3903   
3        True        False

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# ---------------------------
# 1️⃣ Load encoded dataset
# ---------------------------
df = pd.read_csv("encoded_logs.csv")

print("Rows:", len(df))

# ---------------------------
# 2️⃣ Separate target
# ---------------------------
y = df["is_error"]

# ❌ remove leakage columns
X = df.drop(columns=["is_error", "status"])

print("Features used:", list(X.columns))

# ---------------------------
# 3️⃣ Train/Test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ---------------------------
# 4️⃣ Train model
# ---------------------------
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

# ---------------------------
# 5️⃣ Predict
# ---------------------------
pred = model.predict(X_test)

# ---------------------------
# 6️⃣ Evaluation
# ---------------------------
print("\nAccuracy:", accuracy_score(y_test, pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred))

# ---------------------------
# 7️⃣ Save model
# 4_train.py - ADD AT THE END

# Save model bundle with preprocessing info
bundle = {
    "model": model,
    "features": list(X.columns),
    "expected_methods": df.filter(regex='^method_').columns.tolist(),
    "expected_protocols": df.filter(regex='^protocol_').columns.tolist()
}

joblib.dump(bundle, "anomaly_model.pkl")
print(f"\n✓ Saved bundle with {len(bundle['features'])} features")
print("Methods:", bundle['expected_methods'])
print("Protocols:", bundle['expected_protocols'])

Rows: 99768
Features used: ['bytes', 'hour', 'day', 'url_len', 'url_depth', 'is_image', 'is_html', 'method_GET', 'method_HEAD', 'method_POST', 'protocol_HTTP/1.0', 'ip_enc', 'url_enc']

Accuracy: 0.9988473489024757

Confusion Matrix:
[[19851     2]
 [   21    80]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19853
           1       0.98      0.79      0.87       101

    accuracy                           1.00     19954
   macro avg       0.99      0.90      0.94     19954
weighted avg       1.00      1.00      1.00     19954


✓ Saved bundle with 13 features
Methods: ['method_GET', 'method_HEAD', 'method_POST']
Protocols: ['protocol_HTTP/1.0']
